# Final Project: Diabetes Risk Screening
## Dataset Preparation, Model Training and Evaluation

**Name:** TORRECAMPO  
**Course:** CPE ELEC 1 - CpE Elective 1

**Dataset:** CDC Diabetes Health Indicators (UCI Machine Learning Repository, dataset 891)  
**File used:** `diabetes_binary_health_indicators_BRFSS2015.csv`  
**Task:** Binary classification (0 = no diabetes, 1 = prediabetes or diabetes)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, RocCurveDisplay

PROJECT = '/content/drive/MyDrive/CPE ELEC 1 TORRECAMPO/FINAL PROJECT TORRECAMPO/'
CSV_FILE = PROJECT + 'data/diabetes_binary_health_indicators_BRFSS2015.csv'
os.makedirs(PROJECT + 'model', exist_ok=True)
os.makedirs(PROJECT + 'results', exist_ok=True)

## 2. Load Dataset

In [ ]:
df = pd.read_csv(CSV_FILE)
df.shape

In [ ]:
df.head()

In [ ]:
print('Original records:', len(df))
print('Columns:', len(df.columns))
assert len(df) > 5000, 'Dataset must have more than 5,000 records'
assert 'Diabetes_binary' in df.columns, 'Wrong file: target column Diabetes_binary not found'

## 3. Inspect Data

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe().T

In [ ]:
counts = df['Diabetes_binary'].value_counts()
percent = df['Diabetes_binary'].value_counts(normalize=True) * 100
pd.DataFrame({'count': counts, 'percent': percent.round(2)})

## 4. Data Preparation

In [ ]:
records_before = len(df)
df = df.drop_duplicates()
records_after = len(df)
print('Before removing duplicates:', records_before)
print('After removing duplicates:', records_after)
print('Duplicates removed:', records_before - records_after)

The input columns kept for the model are the ones a person can answer without lab tests or insurance details. All values are already numeric (0/1, small scales or counts), so no encoding is needed. There are no missing values.

In [ ]:
FEATURES = ['HighBP', 'HighChol', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
            'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'GenHlth',
            'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age']
TARGET = 'Diabetes_binary'
X = df[FEATURES]
y = df[TARGET].astype(int)
X.shape, y.shape

In [ ]:
print('BMI range:', X['BMI'].min(), 'to', X['BMI'].max())
print('Age groups:', sorted(X['Age'].unique()))
print('GenHlth values:', sorted(X['GenHlth'].unique()))

## 5. Exploratory Charts

In [ ]:
class_counts = y.value_counts().sort_index()
plt.bar(['No diabetes', 'Diabetes'], class_counts.values)
plt.ylabel('Number of records')
plt.title('Class Balance')
plt.savefig(PROJECT + 'results/class_balance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
rate_by_health = df.groupby('GenHlth')['Diabetes_binary'].mean() * 100
plt.bar(rate_by_health.index.astype(int), rate_by_health.values)
plt.xlabel('General health (1 = excellent, 5 = poor)')
plt.ylabel('Diabetes rate (%)')
plt.title('Diabetes Rate by General Health')
plt.savefig(PROJECT + 'results/diabetes_by_general_health.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Diabetes share in train:', round(y_train.mean() * 100, 2), '%')
print('Diabetes share in test:', round(y_test.mean() * 100, 2), '%')

## 7. Evaluation Helper

In [ ]:
def get_scores(model_name, model):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, prob),
    }

## 8. Baseline Model

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_scores = get_scores('Baseline (most frequent)', baseline)
baseline_scores

## 9. Logistic Regression

In [ ]:
log_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000)),
])
log_reg.fit(X_train, y_train)
log_scores = get_scores('Logistic Regression', log_reg)
log_scores

## 10. Random Forest

In [ ]:
rand_forest = Pipeline([
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=50,
                                          class_weight='balanced_subsample', random_state=42, n_jobs=-1)),
])
rand_forest.fit(X_train, y_train)
forest_scores = get_scores('Random Forest', rand_forest)
forest_scores

## 11. Compare Models

In [ ]:
results = pd.DataFrame([baseline_scores, log_scores, forest_scores])
results = results.round(4)
results

In [ ]:
compare = results.set_index('Model')[['Precision', 'Recall', 'F1', 'ROC-AUC']]
compare.plot(kind='bar', figsize=(9, 5))
plt.ylabel('Score')
plt.title('Model Comparison on the Test Set')
plt.xticks(rotation=15)
plt.savefig(PROJECT + 'results/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Select Best Model

In [ ]:
trained_models = {'Logistic Regression': log_reg, 'Random Forest': rand_forest}
candidates = results[results['Model'] != 'Baseline (most frequent)']
best_name = candidates.sort_values('ROC-AUC', ascending=False).iloc[0]['Model']
best_model = trained_models[best_name]
print('Selected model:', best_name)

In [ ]:
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, best_pred, target_names=['No diabetes', 'Diabetes']))

In [ ]:
cm = confusion_matrix(y_test, best_pred)
ConfusionMatrixDisplay(cm, display_labels=['No diabetes', 'Diabetes']).plot(values_format='d')
plt.title('Confusion Matrix: ' + best_name)
plt.savefig(PROJECT + 'results/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
cm

In [ ]:
RocCurveDisplay.from_predictions(y_test, best_prob)
plt.title('ROC Curve: ' + best_name)
plt.savefig(PROJECT + 'results/roc_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## 13. Feature Importance

In [ ]:
final_step = best_model.named_steps['classifier']
if best_name == 'Logistic Regression':
    importance = np.abs(final_step.coef_[0])
else:
    importance = final_step.feature_importances_
importance_table = pd.DataFrame({'Feature': FEATURES, 'Importance': importance}).sort_values('Importance')
plt.barh(importance_table['Feature'], importance_table['Importance'])
plt.xlabel('Importance')
plt.title('Most Important Inputs: ' + best_name)
plt.savefig(PROJECT + 'results/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
importance_table.sort_values('Importance', ascending=False)

## 14. Save Model and Results

In [ ]:
joblib.dump(best_model, PROJECT + 'model/diabetes_model.joblib', compress=3)

model_info = {
    'project': 'Diabetes Risk Screening',
    'dataset': {
        'name': 'CDC Diabetes Health Indicators',
        'source': 'UCI Machine Learning Repository, dataset 891 (BRFSS 2015)',
        'original_records': int(records_before),
        'records_after_cleaning': int(records_after),
    },
    'target': TARGET,
    'features': FEATURES,
    'selected_model': best_name,
    'threshold': 0.5,
    'test_size': 0.20,
    'random_state': 42,
    'sklearn_version': sklearn.__version__,
    'results': results.to_dict(orient='records'),
}
with open(PROJECT + 'model/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

results.to_csv(PROJECT + 'results/model_results.csv', index=False)
with open(PROJECT + 'results/model_results.md', 'w') as f:
    f.write(results.to_markdown(index=False))

with open(PROJECT + 'requirements.txt', 'w') as f:
    f.write('streamlit\npandas\nnumpy\nscikit-learn==' + sklearn.__version__ + '\njoblib\nmatplotlib\n')

## 15. Test the Saved Model

In [ ]:
saved_model = joblib.load(PROJECT + 'model/diabetes_model.joblib')
sample = X_test.head(5)
print(saved_model.predict(sample))
print(saved_model.predict_proba(sample)[:, 1].round(3))
print(y_test.head(5).values)